# Sector Normalization

This notebook normalizes industry sectors and classifications to standard taxonomies including NAICS (North American Industry Classification System), GICS (Global Industry Classification Standard), and custom taxonomies.

## Pipeline Flow
- Input: Industry/sector information from job postings in `workspace.silver.silver_jobs_current`
- Output: Sector mappings written to `workspace.semantic.sem_sector_map`
- Review Queue: Ambiguous sectors in `workspace.silver.silver_semantic_review_queue`
- Taxonomy: NAICS taxonomy maintained in-memory (could be moved to reference table)

## Standard Taxonomies
1. **NAICS**: 2-6 digit hierarchical codes (e.g., 541511 = Custom Computer Programming Services)
2. **GICS**: 8-digit codes organized in 4 levels (Sector, Industry Group, Industry, Sub-Industry)
3. **SIC**: Standard Industrial Classification (legacy)
4. **Custom**: Organization-specific taxonomies

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, ArrayType, TimestampType
from pyspark.sql.window import Window
from datetime import datetime
import re

# Configuration
CONFIG = {
    "input_table": "workspace.silver.silver_jobs_current",
    "output_table": "workspace.semantic.sem_sector_map",
    "review_queue_table": "workspace.silver.silver_semantic_review_queue",
    "primary_taxonomy": "NAICS",  # Primary taxonomy to use
    "confidence_threshold": 0.8,
    "enable_hierarchical_rollup": True,  # Roll up to parent categories if needed
    "max_taxonomy_level": 4
}

In [0]:
# Load sector taxonomy from governed metadata files
# This replaces hardcoded tech-centric NAICS samples with sector-agnostic taxonomy

METADATA_BASE = "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata"

print("Loading sector taxonomy from metadata...")
sectors_raw_df = spark.read.csv(
    f"{METADATA_BASE}/sectors.csv",
    header=True,
    inferSchema=True
)

# Transform to match existing schema expectations
# Map metadata structure to NAICS-compatible structure for backward compatibility
sector_taxonomy_df = sectors_raw_df.select(
    F.coalesce(F.col("naics_code"), F.col("sector_key")).alias("sector_code"),
    F.col("sector_name"),
    F.lit("NAICS").alias("taxonomy_type"),
    F.when(F.col("parent_sector").isNull(), 1).otherwise(2).alias("level"),
    F.col("parent_sector").alias("parent_code"),
    F.split(F.col("keywords"), "\\|").alias("keywords")
)


In [0]:
# Load sector assignments from silver layer
# silver_sector_assign writes sector data to silver_jobs_current
print("Loading sector assignments from silver_jobs_current...")

try:
    jobs_df = spark.table(CONFIG["input_table"])
    
    extracted_sectors_df = jobs_df.select(
        F.col("enterprise_job_id").alias("extraction_id"),
        F.col("sector_assigned").alias("extracted_sector"),
        F.col("sector_assignment_method").alias("assignment_method"),
        F.col("sector_confidence").alias("input_confidence"),
        F.col("enterprise_job_id").alias("source_id")
    ).filter(
        F.col("sector_assigned").isNotNull()
    )
    
    extracted_count = extracted_sectors_df.count()
    print(f"Loaded {extracted_count} sector assignments from silver layer")
    
    if extracted_count == 0:
        print("Warning: No sector assignments found. Run silver_sector_assign first.")
        dbutils.notebook.exit({"status": "no_data", "message": "No sectors to normalize"})
        
except Exception as e:
    print(f"Error loading from silver layer: {e}")
    dbutils.notebook.exit({"status": "error", "message": str(e)})

display(extracted_sectors_df.limit(10))

In [0]:
def normalize_sector_name(name):
    """Normalize sector name for matching."""
    if not name:
        return ""
    
    # Convert to lowercase
    name = name.lower().strip()
    
    # Remove common words
    stop_words = ["and", "the", "of", "in", "for", "services", "industry"]
    words = name.split()
    words = [w for w in words if w not in stop_words]
    
    return " ".join(words)

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

normalize_udf = udf(normalize_sector_name, StringType())

# Normalize extracted sectors
extracted_normalized_df = extracted_sectors_df.withColumn(
    "normalized_sector",
    normalize_udf(F.col("extracted_sector"))
)

print("Normalized extracted sectors:")
display(extracted_normalized_df)

In [0]:
from pyspark.sql.functions import explode, array

# Explode taxonomy keywords for matching
taxonomy_keywords_df = sector_taxonomy_df.select(
    "sector_code",
    "sector_name",
    "taxonomy_type",
    "level",
    "parent_code",
    explode("keywords").alias("keyword")
).withColumn(
    "normalized_keyword",
    normalize_udf(F.col("keyword"))
)

print(f"Expanded taxonomy to {taxonomy_keywords_df.count()} keyword entries")
display(taxonomy_keywords_df.limit(20))

In [0]:
# Match extracted sectors against taxonomy keywords
exact_matches_df = extracted_normalized_df.join(
    taxonomy_keywords_df,
    extracted_normalized_df.normalized_sector == taxonomy_keywords_df.normalized_keyword,
    "left"
).select(
    extracted_normalized_df.extraction_id,
    extracted_normalized_df.extracted_sector,
    extracted_normalized_df.source_id,
    extracted_normalized_df.normalized_sector,
    F.col("sector_code"),
    F.col("sector_name"),
    F.col("taxonomy_type"),
    F.col("level"),
    F.col("parent_code"),
    F.lit("exact_keyword").alias("match_method"),
    F.lit(1.0).alias("confidence")
)

matched_df = exact_matches_df.filter(F.col("sector_code").isNotNull())
unmatched_df = exact_matches_df.filter(F.col("sector_code").isNull()).drop(
    "sector_code", "sector_name", "taxonomy_type", "level", "parent_code", "match_method", "confidence"
)

matched_count = matched_df.count()
unmatched_count = unmatched_df.count()

print(f"Stage 1 - Exact Keyword Match Results:")
print(f"  Matched: {matched_count}")
print(f"  Unmatched: {unmatched_count}")

if matched_count > 0:
    print("\nExact keyword matches:")
    display(matched_df.orderBy("extraction_id"))

In [0]:
# Partial keyword matching using contains
partial_matches_df = unmatched_df.crossJoin(
    taxonomy_keywords_df.select(
        "sector_code", "sector_name", "taxonomy_type", "level", "parent_code", "normalized_keyword"
    ).distinct()
).filter(
    (F.col("normalized_sector").contains(F.col("normalized_keyword"))) |
    (F.col("normalized_keyword").contains(F.col("normalized_sector")))
).select(
    "extraction_id",
    "extracted_sector",
    "source_id",
    "normalized_sector",
    "sector_code",
    "sector_name",
    "taxonomy_type",
    "level",
    "parent_code",
    F.lit("partial_keyword").alias("match_method"),
    F.lit(0.85).alias("confidence")
)

# Keep best match (most specific = highest level number)
if partial_matches_df.count() > 0:
    window_spec = Window.partitionBy("extraction_id").orderBy(F.desc("level"), F.desc("confidence"))
    
    partial_best_df = partial_matches_df.withColumn(
        "rank",
        F.row_number().over(window_spec)
    ).filter(
        F.col("rank") == 1
    ).drop("rank")
    
    partial_matched_count = partial_best_df.count()
    partial_unmatched_count = unmatched_count - partial_matched_count
    
    print(f"\nStage 2 - Partial Keyword Match Results:")
    print(f"  Matched: {partial_matched_count}")
    print(f"  Unmatched: {partial_unmatched_count}")
    
    if partial_matched_count > 0:
        print("\nPartial keyword matches:")
        display(partial_best_df)
    
    # Get remaining unmatched
    final_unmatched_df = unmatched_df.join(
        partial_best_df.select("extraction_id"),
        "extraction_id",
        "left_anti"
    )
    
    # Combine all matches
    all_matched_df = matched_df.union(partial_best_df)
else:
    print(f"\nStage 2 - No partial matches found")
    final_unmatched_df = unmatched_df
    all_matched_df = matched_df

In [0]:
# Optionally roll up to parent categories for more general classification
if CONFIG["enable_hierarchical_rollup"]:
    print(f"\nApplying hierarchical rollup to level {CONFIG['max_taxonomy_level']}...")
    
    # For matches that are too detailed, roll up to parent
    rollup_df = all_matched_df.filter(
        F.col("level") > CONFIG["max_taxonomy_level"]
    )
    
    if rollup_df.count() > 0:
        # Join with parent codes to get rolled-up sectors
        rolled_up_df = rollup_df.join(
            sector_taxonomy_df.select(
                F.col("sector_code").alias("parent_sector_code"),
                F.col("sector_name").alias("parent_sector_name"),
                F.col("level").alias("parent_level")
            ),
            rollup_df.parent_code == F.col("parent_sector_code"),
            "left"
        ).select(
            "extraction_id",
            "extracted_sector",
            "source_id",
            "normalized_sector",
            F.col("parent_sector_code").alias("sector_code"),
            F.col("parent_sector_name").alias("sector_name"),
            F.col("taxonomy_type"),
            F.col("parent_level").alias("level"),
            F.lit(None).alias("parent_code"),
            F.lit("hierarchical_rollup").alias("match_method"),
            F.col("confidence") * 0.9
        )
        
        # Replace rolled-up records in matched set
        keep_df = all_matched_df.filter(F.col("level") <= CONFIG["max_taxonomy_level"])
        all_matched_df = keep_df.union(rolled_up_df)
        
        print(f"Rolled up {rollup_df.count()} records to parent categories")
else:
    print("\nHierarchical rollup disabled")

In [0]:
# Split by confidence threshold
high_confidence_df = enriched_df.filter(
    F.col("confidence") >= CONFIG["confidence_threshold"]
).withColumn(
    "processed_timestamp",
    F.current_timestamp()
)

low_confidence_df = enriched_df.filter(
    F.col("confidence") < CONFIG["confidence_threshold"]
).withColumn(
    "review_reason",
    F.lit("low_confidence")
).withColumn(
    "review_status",
    F.lit("pending")
).withColumn(
    "processed_timestamp",
    F.current_timestamp()
)

# Add unmatched to review queue
unmatched_review_df = final_unmatched_df.select(
    "extraction_id",
    "extracted_sector",
    "source_id",
    "normalized_sector",
    F.lit(None).cast(StringType()).alias("sector_code"),
    F.lit(None).cast(StringType()).alias("sector_name"),
    F.lit(CONFIG["primary_taxonomy"]).alias("taxonomy_type"),
    F.lit(None).cast(IntegerType()).alias("level"),
    F.lit(None).cast(StringType()).alias("parent_code"),
    F.lit("no_match").alias("match_method"),
    F.lit(0.0).alias("confidence"),
    F.lit("no_match").alias("review_reason"),
    F.lit("pending").alias("review_status"),
    F.current_timestamp().alias("processed_timestamp")
)

review_queue_df = low_confidence_df.union(unmatched_review_df)

high_conf_count = high_confidence_df.count()
review_count = review_queue_df.count()

print(f"\nNormalization Summary:")
print(f"  High confidence (>= {CONFIG['confidence_threshold']}): {high_conf_count}")
print(f"  Review queue: {review_count}")
print(f"  Success rate: {high_conf_count / extracted_count * 100:.1f}%")

In [0]:
# Enrich matched sectors with hierarchy information for warehouse consumption
# Create sector hierarchy mapping (category, level, parent relationships)

# Build sector hierarchy reference
sector_hierarchy = [
    # (canonical_name, category, level, parent)
    ("Information", "Technology", 1, None),
    ("Professional, Scientific, and Technical Services", "Professional Services", 1, None),
    ("Health Care and Social Assistance", "Healthcare", 1, None),
    ("Finance and Insurance", "Finance", 1, None),
    ("Retail Trade", "Retail", 1, None),
    ("Manufacturing", "Manufacturing", 1, None),
    ("Educational Services", "Education", 1, None),
    ("Software Publishers", "Technology", 2, "Information"),
    ("Computer Systems Design", "Technology", 2, "Information"),
]

hierarchy_df = spark.createDataFrame(
    sector_hierarchy,
    ["sector_name", "sector_category", "sector_level", "parent_sector"]
)

print("Sector hierarchy reference built")
print(f"Hierarchical mappings: {hierarchy_df.count()}")

# Enrich matched sectors with hierarchy
enriched_df = all_matched_df.join(
    hierarchy_df,
    all_matched_df.sector_name == hierarchy_df.sector_name,
    "left"
).select(
    all_matched_df["*"],
    hierarchy_df["sector_category"],
    F.coalesce(hierarchy_df["sector_level"], F.lit(1)).alias("sector_level_hierarchy"),
    hierarchy_df["parent_sector"]
)

print("Sectors enriched with hierarchy information")
display(enriched_df.limit(10))

In [0]:
# Prepare data for sem_sector_map schema
from pyspark.sql.functions import md5, concat, lit, when

output_df = high_confidence_df.select(
    md5(concat(F.col("extraction_id"), F.col("sector_code"))).alias("sector_map_id"),
    F.col("extraction_id").alias("enterprise_job_id"),
    F.col("extracted_sector").alias("sector_name_norm"),  # Original sector from Silver
    F.col("sector_name").alias("canonical_sector_name"),  # Normalized NAICS sector
    F.coalesce(F.col("sector_category"), F.lit("Uncategorized")).alias("sector_category"),
    F.col("sector_level_hierarchy").alias("sector_level"),
    F.col("parent_sector"),
    F.lit(True).alias("is_active"),
    when(F.col("match_method") == "exact_keyword", "EXACT_MATCH")
     .when(F.col("match_method") == "partial_keyword", "FUZZY")
     .when(F.col("match_method") == "hierarchical_rollup", "ROLLUP")
     .otherwise("MANUAL").alias("normalization_method"),
    F.col("confidence").cast("decimal(5,4)").alias("normalization_confidence"),
    F.to_json(F.struct(
        F.col("extracted_sector").alias("input"),
        F.col("normalized_sector").alias("normalized"),
        F.col("match_method").alias("method"),
        F.col("level").alias("taxonomy_level")
    )).alias("explanation_json"),
    F.date_format(F.current_timestamp(), "yyyyMMdd_HHmmss").alias("effective_batch_id")
)

# Write to semantic.sem_sector_map
print(f"\nWriting {high_conf_count} sector mappings to {CONFIG['output_table']}...")

output_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(CONFIG["output_table"])

print("✓ Sector mappings written successfully")

# Write review queue
if review_count > 0:
    print(f"\nWriting {review_count} records to review queue at {CONFIG['review_queue_table']}...")
    
    review_df = review_queue_df.select(
        F.expr("uuid()").alias("review_id"),
        F.col("extraction_id").alias("enterprise_job_id"),
        F.col("review_reason").alias("issue_type"),
        F.concat(
            F.lit("Sector: "),
            F.col("extracted_sector"),
            F.lit(", Normalized: "),
            F.col("normalized_sector"),
            F.lit(", Suggested: "),
            F.coalesce(F.col("sector_name"), F.lit("none"))
        ).alias("issue_detail"),
        F.coalesce(F.col("confidence"), F.lit(0.0)).cast("decimal(5,4)").alias("confidence"),
        F.current_timestamp().alias("queued_at"),
        F.col("review_status").alias("status")
    )
    
    review_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(CONFIG["review_queue_table"])
    
    print("✓ Review queue records written successfully")

In [0]:
# Analyze sector distribution by taxonomy level
level_dist_df = high_confidence_df.groupBy("level", "taxonomy_type").agg(
    F.count("*").alias("count")
).orderBy("level")

print("\nSector Distribution by Taxonomy Level:")
display(level_dist_df)

# Top sectors
top_sectors_df = high_confidence_df.groupBy("sector_code", "sector_name").agg(
    F.count("*").alias("count")
).orderBy(F.desc("count")).limit(10)

print("\nTop 10 Normalized Sectors:")
display(top_sectors_df)

# Match method breakdown
method_stats_df = high_confidence_df.groupBy("match_method").agg(
    F.count("*").alias("count"),
    F.avg("confidence").alias("avg_confidence")
).orderBy(F.desc("count"))

print("\nMatch Method Statistics:")
display(method_stats_df)

print("\n" + "="*60)
print("SECTOR NORMALIZATION - EXECUTION SUMMARY")
print("="*60)
print(f"Input sectors: {extracted_count}")
print(f"Successfully normalized: {high_conf_count} ({high_conf_count/extracted_count*100:.1f}%)")
print(f"Review queue: {review_count} ({review_count/extracted_count*100:.1f}%)")
print(f"Primary taxonomy: {CONFIG['primary_taxonomy']}")
print(f"Confidence threshold: {CONFIG['confidence_threshold']}")
print("="*60)